In [ ]:
#| default_exp pipeline

# pipeline

> Top-level pipeline orchestrator.
>
> `run()` wires all steps together: analyze → storyboard → validate (optional)
> → character sheets (capability-gated) → render panels → return ComicOutput.
>
> Individual steps can also be invoked directly via `run_analyze_only()`,
> `run_storyboard_only()`, and `run_render_only()` for incremental workflows.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import asyncio
import time
from pathlib import Path

from rich.console import Console
from rich.rule import Rule

from manhualizer.analyze import analyze_story
from manhualizer.character_sheets import generate_character_sheets_async
from manhualizer.config import PipelineConfig
from manhualizer.llm import LLMClient
from manhualizer.models import ComicOutput, StoryAnalysis, Storyboard
from manhualizer.prompts import load_templates
from manhualizer.render import get_renderer
from manhualizer.storyboard import build_storyboard
from manhualizer.validate import validate_storyboard

_console = Console()


In [ ]:
#| export
def _output_dir(config: PipelineConfig, story_path: Path) -> Path:
    """Resolve the output directory, defaulting to <story_stem>_comic/ next to the story."""
    if config.output.dir != "output":
        return Path(config.output.dir)
    return story_path.parent / f"{story_path.stem}_comic"

In [ ]:
#| export
def run(
    story_path: str | Path,
    config: PipelineConfig | None = None,
) -> ComicOutput:
    """Run the full manhualizer pipeline.

    Steps:
    1. Analyze story text → analysis.json
    2. Build storyboard → storyboard.json
    3. Validate storyboard (optional, config.run_validation)
    4. Generate character sheets (only if model supports reference_images)
    5. Render all panels concurrently → panels/

    All intermediate results are persisted to disk. If `config.resume` is True
    (the default), completed steps are skipped on re-runs.

    Args:
        story_path: Path to the story text file.
        config: PipelineConfig. If None, defaults are used (looks for
                manhualizer.yml in the current directory).

    Returns:
        ComicOutput with paths to all generated files.
    """
    story_path = Path(story_path)
    if not story_path.exists():
        raise FileNotFoundError(f"Story file not found: {story_path}")

    if config is None:
        from manhualizer.config import load_config
        config = load_config()

    output_dir = _output_dir(config, story_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    story_text = story_path.read_text(encoding="utf-8")

    templates = load_templates(config.template, config.custom_templates_dir)
    llm = LLMClient(config.llm, templates)
    renderer = get_renderer(config.renderer.model, config.renderer)

    total_steps = 3 + (1 if config.run_validation else 0) + (1 if renderer.model_spec.capabilities.reference_images else 0)
    step = 0

    def _step(label: str) -> float:
        nonlocal step
        step += 1
        _console.print(f"[bold][{step}/{total_steps}][/bold] {label}")
        return time.monotonic()

    def _done(t0: float, detail: str = "") -> None:
        elapsed = time.monotonic() - t0
        suffix = f"  [dim]{detail}[/dim]" if detail else ""
        _console.print(f"    [green]done[/green] in {elapsed:.1f}s{suffix}")

    _console.print(Rule(f"[bold]manhualizer[/bold] — {story_path.name}"))
    _console.print(f"  model: [cyan]{config.renderer.model}[/cyan]  "
                   f"template: [cyan]{config.template}[/cyan]  "
                   f"output: [cyan]{output_dir}[/cyan]")

    # ── Step 1: Analyze ──────────────────────────────────────────────────────
    t0 = _step("Analyzing story…")
    analysis = analyze_story(
        story_text, llm, config, output_dir / "analysis.json"
    )
    _done(t0, f"{len(analysis.characters)} character(s), {len(analysis.locations)} location(s)")

    # ── Step 2: Storyboard ───────────────────────────────────────────────────
    t0 = _step("Building storyboard…")
    storyboard = build_storyboard(
        story_text, analysis, llm, templates, config, output_dir / "storyboard.json"
    )
    panel_count = len(storyboard.all_panels)
    _done(t0, f"{len(storyboard.scenes)} scene(s), {panel_count} panel(s)")

    # ── Step 3: Validate (optional) ──────────────────────────────────────────
    validation_path: Path | None = None
    if config.run_validation:
        t0 = _step("Validating storyboard…")
        validation_path = output_dir / "validation.json"
        validate_storyboard(story_text, storyboard, llm, config, validation_path)
        _done(t0)

    # ── Step 4: Character sheets (capability-gated) ───────────────────────────
    character_sheets: dict[str, Path] = {}
    if renderer.model_spec.capabilities.reference_images:
        t0 = _step("Generating character sheets…")
        character_sheets = asyncio.run(
            generate_character_sheets_async(
                analysis, renderer,
                output_dir / "character_sheets",
                templates, config,
            )
        )
        _done(t0, f"{len(character_sheets)} sheet(s)")

    # ── Step 5: Render panels ─────────────────────────────────────────────────
    t0 = _step(f"Rendering {panel_count} panel(s)…")
    panels_dir = output_dir / "panels"
    render_results = asyncio.run(
        renderer.render_batch_async(
            storyboard.all_panels,
            panels_dir,
            config.output,
            reference_images=character_sheets or None,
            resume=config.resume,
        )
    )
    _done(t0)

    result = ComicOutput(
        output_dir=output_dir,
        analysis_path=output_dir / "analysis.json",
        storyboard_path=output_dir / "storyboard.json",
        validation_path=validation_path,
        rendered_panels=render_results,
    )

    _console.print(Rule())
    _console.print(f"[bold green]Done![/bold green] {len(render_results)} panel(s) → {output_dir}")
    return result


In [ ]:
#| export
def run_analyze_only(
    story_path: str | Path,
    config: PipelineConfig | None = None,
) -> StoryAnalysis:
    """Run only the analysis step and return the StoryAnalysis."""
    story_path = Path(story_path)
    if config is None:
        from manhualizer.config import load_config
        config = load_config()
    output_dir = _output_dir(config, story_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    templates = load_templates(config.template, config.custom_templates_dir)
    llm = LLMClient(config.llm, templates)
    return analyze_story(
        story_path.read_text(encoding="utf-8"), llm, config, output_dir / "analysis.json"
    )


def run_storyboard_only(
    story_path: str | Path,
    config: PipelineConfig | None = None,
) -> Storyboard:
    """Run analysis + storyboard steps. Resumes from analysis.json if available."""
    story_path = Path(story_path)
    if config is None:
        from manhualizer.config import load_config
        config = load_config()
    output_dir = _output_dir(config, story_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    templates = load_templates(config.template, config.custom_templates_dir)
    llm = LLMClient(config.llm, templates)
    story_text = story_path.read_text(encoding="utf-8")
    analysis = analyze_story(story_text, llm, config, output_dir / "analysis.json")
    return build_storyboard(story_text, analysis, llm, templates, config, output_dir / "storyboard.json")


def run_render_only(
    story_path: str | Path,
    config: PipelineConfig | None = None,
) -> ComicOutput:
    """Run render step only. Requires analysis.json and storyboard.json to exist."""
    story_path = Path(story_path)
    if config is None:
        from manhualizer.config import load_config
        config = load_config()
    output_dir = _output_dir(config, story_path)

    analysis_path = output_dir / "analysis.json"
    storyboard_path = output_dir / "storyboard.json"
    for p in (analysis_path, storyboard_path):
        if not p.exists():
            raise FileNotFoundError(
                f"{p} not found. Run `manhualizer storyboard-only` first."
            )

    templates = load_templates(config.template, config.custom_templates_dir)
    analysis = StoryAnalysis.model_validate_json(analysis_path.read_text())
    storyboard = Storyboard.model_validate_json(storyboard_path.read_text())
    renderer = get_renderer(config.renderer.model, config.renderer)

    character_sheets: dict[str, Path] = {}
    if renderer.model_spec.capabilities.reference_images:
        character_sheets = asyncio.run(
            generate_character_sheets_async(
                analysis, renderer, output_dir / "character_sheets", templates, config
            )
        )

    render_results = asyncio.run(
        renderer.render_batch_async(
            storyboard.all_panels, output_dir / "panels",
            config.output, reference_images=character_sheets or None,
            resume=config.resume,
        )
    )
    return ComicOutput(
        output_dir=output_dir,
        analysis_path=analysis_path,
        storyboard_path=storyboard_path,
        rendered_panels=render_results,
    )

## Tests (no API calls)

In [ ]:
import tempfile, json
from pathlib import Path
from manhualizer.pipeline import _output_dir
from manhualizer.config import PipelineConfig

# Default output dir: next to story file
cfg = PipelineConfig()
story = Path("/tmp/my_story.txt")
out = _output_dir(cfg, story)
assert out == Path("/tmp/my_story_comic")

# Explicit output dir overrides default
cfg2 = PipelineConfig()
cfg2.output.dir = "/custom/output"
out2 = _output_dir(cfg2, story)
assert out2 == Path("/custom/output")

print("Output dir resolution OK")

In [ ]:
# End-to-end pipeline smoke test with stub LLM + stub renderer
import asyncio, json, tempfile
from pathlib import Path
from manhualizer.config import PipelineConfig, RendererConfig
from manhualizer.models import (
    Character, Location, StoryAnalysis, Scene, Panel, Storyboard, RenderResult
)
from manhualizer.render import BaseRenderer, MODELS
from manhualizer.llm import LLMClient
from manhualizer.prompts import load_templates

# Stub LLM that returns pre-baked JSON without calling any API
class StubLLM(LLMClient):
    def __init__(self, analysis_data, storyboard_data):
        self._analysis_data = analysis_data
        self._storyboard_data = storyboard_data
        self._call_count = 0
    def complete_from_template(self, template_file, prompt_key, as_json=False, **kwargs):
        self._call_count += 1
        if template_file == "analyze.yml":
            return self._analysis_data
        if template_file == "storyboard.yml":
            return self._storyboard_data
        return {}

class StubRenderer(BaseRenderer):
    async def render_async(self, panel, output_dir, output_cfg, reference_images=None):
        path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        path.write_bytes(b"stub_image")
        return RenderResult(panel_number=panel.panel_number, image_path=path,
                            backend_used="stub", prompt_used=panel.visual_prompt)

analysis_data = {
    "title": "The Dragon's Gift",
    "synopsis": "A young farmer discovers a dragon.",
    "characters": [{"name": "Wei Chen", "aliases": [],
                    "physical_description": "Tall young man", "personality": "Brave",
                    "reference_image_prompt": "young man black hair", "arcs": []}],
    "locations": [{"name": "Village", "description": "Small village",
                   "visual_prompt": "chinese village", "atmosphere": "warm"}],
    "themes": ["courage"],
}
storyboard_data = {
    "scene_id": "s1", "title": "Discovery",
    "panels": [
        {"visual_prompt": "manhua style, village, Wei Chen walks",
         "characters_present": ["Wei Chen"], "location": "Village",
         "action_description": "walks", "mood": "peaceful",
         "camera_angle": "wide shot", "dialogue": []},
        {"visual_prompt": "manhua style, cave, Wei Chen finds egg",
         "characters_present": ["Wei Chen"], "location": "Cave",
         "action_description": "finds egg", "mood": "mysterious",
         "camera_angle": "close-up", "dialogue": []},
    ]
}

with tempfile.TemporaryDirectory() as tmp:
    story_file = Path(tmp) / "story.txt"
    story_file.write_text("Once upon a time, Wei Chen found a dragon's egg.")

    cfg = PipelineConfig(resume=False)
    cfg.output.dir = str(Path(tmp) / "output")

    # Patch the pipeline to use stubs
    import manhualizer.pipeline as pipeline_mod
    original_llm = pipeline_mod.LLMClient
    original_renderer = pipeline_mod.get_renderer

    stub_llm = StubLLM(analysis_data, storyboard_data)
    stub_renderer = StubRenderer(MODELS["flux-klein"], RendererConfig())

    pipeline_mod.LLMClient = lambda *a, **kw: stub_llm
    pipeline_mod.get_renderer = lambda *a, **kw: stub_renderer

    try:
        from manhualizer.pipeline import run
        result = run(story_file, cfg)
        assert result.output_dir.exists()
        assert result.analysis_path.exists()
        assert result.storyboard_path.exists()
        assert len(result.rendered_panels) == 2
        assert all(r.image_path.exists() for r in result.rendered_panels)
        assert result.validation_path is None  # run_validation=False
        print(f"Pipeline smoke test OK — {len(result.rendered_panels)} panels rendered")
    finally:
        pipeline_mod.LLMClient = original_llm
        pipeline_mod.get_renderer = original_renderer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()